<a href="https://colab.research.google.com/github/mohamed-nasser11/IMDb-Top-1000-Multi-Modal-Analysis-Image-Text-Classification/blob/main/copy_of_imdb_top_1000_multi_modal_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IMDb Top 1000 Multi-Modal Scraper

This project builds a multi-modal ML pipeline on 1000 IMDb movies scraped across Action, Drama, Comedy, and Thriller genres using Playwright to bypass dynamic rendering and anti-bot mechanisms.


# ________________________________________________________________________________

# Environment Setup

Install required libraries and system dependencies for web scraping using Playwright.

In [ ]:
!pip install requests beautifulsoup4 playwright
!playwright install chromium

In [ ]:
!apt-get install -y \
    libatk1.0-0 \
    libatk-bridge2.0-0 \
    libcups2 \
    libxkbcommon0 \
    libxcomposite1 \
    libxdamage1 \
    libxfixes3 \
    libxrandr2 \
    libgbm1 \
    libasound2 \
    libpango-1.0-0 \
    libpangocairo-1.0-0 \
    libnspr4 \
    libnss3

## Web Scraping IMDb Top 1000

This section implements an asynchronous web scraping pipeline using Playwright to extract structured movie data from IMDb across multiple genre-based pages.

The pipeline performs the following steps:

Launches a headless Chromium browser
Loads multiple IMDb search pages across different genres
Scrolls through each page to ensure all dynamic content is fully loaded
Extracts structured movie data including rank, title, year, rating, votes, genre, and image URL
Handles dynamic rendering and missing or inconsistent elements
Builds a unified structured dataset of up to 1000 movies
Removes duplicates and ensures clean data consistency
Saves the final output as a JSON file for further analysis and machine learning tasks

In [ ]:
import asyncio
from playwright.async_api import async_playwright
import json
import re

async def scrape_imdb_by_genre():
    movies = []
    pages_urls = [
        "https://www.imdb.com/search/title/?title_type=feature&genres=action&sort=user_rating,desc&count=250",
        "https://www.imdb.com/search/title/?title_type=feature&genres=drama&sort=user_rating,desc&count=250",
        "https://www.imdb.com/search/title/?title_type=feature&genres=comedy&sort=user_rating,desc&count=250",
        "https://www.imdb.com/search/title/?title_type=feature&genres=thriller&sort=user_rating,desc&count=250"
    ]

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=[
                "--no-sandbox",
                "--disable-dev-shm-usage",
                "--disable-gpu",
                "--single-process"
            ]
        )

        page = await browser.new_page(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            locale="en-US"
        )

        await page.add_init_script(
            "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
        )

        for page_num, url in enumerate(pages_urls, 1):
            genre = url.split("genres=")[1].split("&")[0]
            print(f"\n--- Page {page_num}/4 | Genre: {genre} ---")

            await page.goto(
                url,
                wait_until="load",
                timeout=60000
            )

            await page.wait_for_timeout(3000)

            for _ in range(10):
                await page.evaluate("window.scrollBy(0, 1500)")
                await page.wait_for_timeout(300)

            await page.wait_for_timeout(2000)

            items = await page.query_selector_all(
                "li.ipc-metadata-list-summary-item"
            )

            print(f"Items found: {len(items)}")

            for i, item in enumerate(items):
                try:
                    title_el = await item.query_selector("h3.ipc-title__text")
                    full_title = await title_el.inner_text() if title_el else ""

                    match = re.match(r"(\d+)\.\s+(.*)", full_title)

                    if match:
                        rank = int(match.group(1))
                        title = match.group(2)
                    else:
                        rank = len(movies) + 1
                        title = full_title

                    link_el = await item.query_selector("a.ipc-lockup-overlay")
                    href = await link_el.get_attribute("href") if link_el else ""

                    tt_match = re.search(r"/(tt\d+)/", href)
                    tt_id = tt_match.group(1) if tt_match else None

                    img_el = await item.query_selector("img.ipc-image")
                    image = await img_el.get_attribute("src") if img_el else ""

                    if not image or not image.startswith("http"):
                        image = await img_el.get_attribute("data-src") if img_el else ""

                    alt = await img_el.get_attribute("alt") if img_el else ""
                    year_match = re.search(r"\((\d{4})\)", alt)
                    year = year_match.group(1) if year_match else None

                    rating_el = await item.query_selector("span.ipc-rating-star--rating")
                    rating = await rating_el.inner_text() if rating_el else None

                    votes_el = await item.query_selector("span.ipc-rating-star--voteCount")
                    votes = await votes_el.inner_text() if votes_el else None

                    if title and image and image.startswith("http"):
                        movies.append({
                            "rank": rank,
                            "title": title,
                            "tt_id": tt_id,
                            "genre": genre,
                            "image": image,
                            "year": year,
                            "rating": rating,
                            "votes": votes
                        })

                        print(f"  [{rank}] {title} | {tt_id} | {year} | {rating}")

                except Exception as e:
                    print(f"  [ERROR] Item {i+1}: {e}")

        await browser.close()

    return movies


movies = await scrape_imdb_by_genre()
print(f"\nDone. Total scraped: {len(movies)} movies")

with open("imdb_by_genre.json", "w", encoding="utf-8") as f:
    json.dump(movies, f, indent=2, ensure_ascii=False)

from google.colab import files
files.download("imdb_by_genre.json")

# Check that all scraped features (rank, title, year, rating, votes, image) are correctly collected and displayed.

In [ ]:
from IPython.display import display, HTML

html = """
<style>
.card {
    display: inline-block;
    width: 220px;
    margin: 10px;
    border: 1px solid #ddd;
    border-radius: 10px;
    overflow: hidden;
    font-family: Arial;
    box-shadow: 2px 2px 10px rgba(0,0,0,0.1);
}

.card img {
    width: 100%;
    height: 320px;
    object-fit: cover;
}

.info {
    padding: 10px;
}

.title {
    font-size: 14px;
    font-weight: bold;
}

.meta {
    font-size: 12px;
    color: #555;
    margin-top: 5px;
}
</style>
"""

display(HTML(html))

# Filter only Drama movies
drama_movies = [movie for movie in movies if movie["genre"] == "drama"]

# Show first 20 Drama movies
for movie in drama_movies[:20]:
    display(HTML(f"""
    <div class="card">
        <img src="{movie['image']}">
        <div class="info">
            <div class="title">#{movie['rank']} {movie['title']}</div>
            <div class="meta">Genre: {movie['genre']}</div>
            <div class="meta">Year: {movie['year']}</div>
            <div class="meta">Rating: ⭐ {movie['rating']}</div>
            <div class="meta">Votes: {movie['votes']}</div>
        </div>
    </div>
    """))

# Movies DataFrame

In [ ]:
import pandas as pd
movies_df = pd.DataFrame(movies)

# Display dataset
display(movies_df)

## Data Cleaning & Preparation

In [ ]:
## Data Cleaning & Preparation
# Convert raw scraped data into a clean, typed DataFrame ready for modeling.
# Steps: remove nulls, fix types, normalize votes from shorthand (K/M) to integers, remove duplicates.

import pandas as pd

df = pd.DataFrame(movies)

# Initial inspection
print("Before cleaning:")
print(df.shape)
print(df.isnull().sum())

# Remove parentheses and whitespace from votes column
df["votes"] = df["votes"].str.replace(r"[\(\)\s]", "", regex=True)

# Convert rating to float and year to numeric, coercing invalid values to NaN
df["rating"] = df["rating"].astype(float)
df["year"] = pd.to_numeric(df["year"], errors="coerce")

# Drop rows where year is missing
df = df.dropna(subset=["year"])
df["year"] = df["year"].astype(int)

# Convert votes from shorthand strings (e.g. 3.2M, 28K) to full integers
def convert_votes(v):
    if pd.isna(v):
        return None
    v = str(v).strip()
    if "M" in v:
        return float(v.replace("M", "")) * 1_000_000
    elif "K" in v:
        return float(v.replace("K", "")) * 1_000
    return float(v)

df["votes"] = df["votes"].apply(convert_votes).astype(int)

# Remove duplicate movies
df = df.drop_duplicates(subset=["tt_id"])

# Final inspection
print("\nAfter cleaning:")
print(df.shape)
print(df.dtypes)
print(df.head())

# Save cleaned dataset
df.to_csv("imdb_clean.csv", index=False)

from google.colab import files
files.download("imdb_clean.csv")

In [ ]:
df = df.drop_duplicates(subset=["tt_id"])
print(f"After dedup: {len(df)} movies")
print(f"Unique tt_ids: {df['tt_id'].nunique()}")

In [ ]:
import os

os.listdir()

In [ ]:
import pandas as pd

df = pd.read_csv("imdb_clean.csv")

df.head()

In [ ]:
df.shape
df.info()

In [ ]:
# Remove rows where year is missing (NaN)
df = df.dropna(subset=["year"])

# Convert year from float to integer for consistency
df["year"] = df["year"].astype(int)

In [ ]:
# Function to convert votes from string format (e.g., "2.3M", "900K") to numeric value
def convert_votes(v):
    if isinstance(v, str):
        v = v.replace("(", "").replace(")", "").replace(",", "")

        # Convert millions (M) to numeric
        if "M" in v:
            return float(v.replace("M", "")) * 1_000_000

        # Convert thousands (K) to numeric
        if "K" in v:
            return float(v.replace("K", "")) * 1_000

    return v

# Apply conversion to the votes column
df["votes"] = df["votes"].apply(convert_votes)

# Ensure final type is float for ML usage
df["votes"] = df["votes"].astype(float)

## Download And Upload Cleaned Data & Posters


In [ ]:
import json
import pandas as pd
from google.colab import files

df = pd.DataFrame(movies)

df.to_csv("imdb_clean.csv", index=False)

df.to_json("imdb_clean.json", orient="records", indent=2)

files.download("imdb_clean.csv")
files.download("imdb_clean.json")

In [ ]:
import os
os.makedirs("posters", exist_ok=True)
print(os.listdir("posters"))

In [ ]:
import shutil
import os

shutil.rmtree("posters")
os.makedirs("posters", exist_ok=True)
print("Deleted and recreated posters folder")

In [ ]:
import asyncio
from playwright.async_api import async_playwright
import os
from tqdm import tqdm

os.makedirs("posters", exist_ok=True)

df["image_lg"] = df["image"].str.replace(
    r"_V1_.*\.jpg",
    "_V1_FMjpg_UX500_.jpg",
    regex=True
)

async def download_posters():
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=["--no-sandbox", "--disable-dev-shm-usage", "--disable-gpu", "--single-process"]
        )

        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
            extra_http_headers={"Referer": "https://www.imdb.com/"}
        )

        for _, row in tqdm(df.iterrows(), total=len(df)):
            filename = f"posters/{row['tt_id']}.jpg"
            if os.path.exists(filename):
                continue
            try:
                response = await context.request.get(row["image_lg"], timeout=10000)
                if response.status == 200:
                    with open(filename, "wb") as f:
                        f.write(await response.body())
            except:
                pass

        await browser.close()

    print(f"Downloaded: {len(os.listdir('posters'))} posters")

await download_posters()

In [ ]:
import os
from PIL import Image

img = Image.open(f"posters/{df['tt_id'].iloc[0]}.jpg")
print(img.size)

In [ ]:
import shutil

shutil.make_archive("posters_dataset", "zip", "posters")

files.download("posters_dataset.zip")

## DATA CLEANING FINAL STEP

## EDA & Visual Analysis

In this section, we explore the dataset visually to better understand its structure, distributions, and potential challenges before modeling.

The analysis covers 7 key aspects:
1. **Rating Distribution** — How IMDb ratings are spread across all movies
2. **Genre Count** — Number of movies per genre and dataset balance
3. **Year vs Rating** — Whether release year correlates with rating
4. **Correlation Heatmap** — Relationships between rating, votes, and year
5. **Votes Distribution** — How popularity varies across movies
6. **Class Imbalance Check** — Whether genres are balanced for fair training
7. **Sample Posters** — Visual inspection of image data used in CNN models

In [ ]:
# Show distribution of IMDb ratings across all movies
import matplotlib.pyplot as plt

plt.hist(df["rating"], bins=20)
plt.title("IMDb Rating Distribution")
plt.xlabel("Rating")
plt.ylabel("Number of Movies")
plt.show()

In [ ]:
# Count how many movies exist in each genre
df["genre"].value_counts().plot(kind="bar")

plt.title("Number of Movies per Genre")
plt.xlabel("Genre")
plt.ylabel("Count")
plt.show()

In [ ]:
# Check how movie ratings change over years
df_clean = df.dropna(subset=["year", "rating"])
df_clean["year"] = df_clean["year"].astype(int)

plt.scatter(df_clean["year"], df_clean["rating"], alpha=0.5)
plt.title("Year vs Rating")
plt.xlabel("Year")
plt.ylabel("Rating")
plt.show()

In [ ]:
# Correlation heatmap between numerical features.
# Votes column still contains raw strings with parentheses (e.g. "1.8K"),
# so we clean it here locally before computing correlations.
import seaborn as sns
import numpy as np

df_corr = df.copy()

def clean_votes_corr(v):
    if pd.isna(v):
        return np.nan
    v = str(v).replace("(", "").replace(")", "").replace(",", "").strip()
    if "M" in v or "m" in v:
        return float(v.replace("M", "").replace("m", "")) * 1_000_000
    elif "K" in v or "k" in v:
        return float(v.replace("K", "").replace("k", "")) * 1_000
    try:
        return float(v)
    except:
        return np.nan

df_corr["votes"] = df_corr["votes"].apply(clean_votes_corr)
df_corr["rating"] = pd.to_numeric(df_corr["rating"], errors="coerce")
df_corr["year"] = pd.to_numeric(df_corr["year"], errors="coerce")
df_corr = df_corr.dropna(subset=["rating", "votes", "year"])

plt.figure(figsize=(6, 4))
sns.heatmap(df_corr[["rating", "votes", "year"]].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Between Numerical Features")
plt.tight_layout()
plt.show()

In [ ]:
# Votes distribution — shows how popular movies are across the dataset.
# Log scale is used because votes range from ~10 to 1M+ (very skewed).
# X-axis ticks are labeled with real numbers for easy reading.
# Y-axis grid lines added to make bar heights easy to read.

import numpy as np

df_votes = df.copy()
df_votes["votes_clean"] = df_votes["votes"].apply(clean_votes_corr)
df_votes = df_votes.dropna(subset=["votes_clean"])

plt.figure(figsize=(10, 5))
counts, edges, patches = plt.hist(df_votes["votes_clean"], bins=30, color="steelblue", edgecolor="black")
plt.xscale("log")

# Real number labels on X-axis
plt.xticks(
    [100, 1_000, 10_000, 100_000, 1_000_000],
    ["100", "1K", "10K", "100K", "1M"],
    fontsize=10
)

# Y-axis grid lines for easy reading
plt.grid(axis="y", alpha=0.4, linestyle="--")

# Label on tallest bar
tallest_idx = counts.argmax()
tallest_x = (edges[tallest_idx] + edges[tallest_idx + 1]) / 2
plt.text(tallest_x, counts[tallest_idx] + 5, str(int(counts[tallest_idx])),
         ha="center", va="bottom", fontweight="bold", color="red", fontsize=11)

plt.title("Votes Distribution Across Movies")
plt.xlabel("Number of Votes")
plt.ylabel("Number of Movies")
plt.tight_layout()
plt.show()

# Summary stats
print(f"Min votes:    {int(df_votes['votes_clean'].min()):,}")
print(f"Max votes:    {int(df_votes['votes_clean'].max()):,}")
print(f"Median votes: {int(df_votes['votes_clean'].median()):,}")

In [ ]:
# Class distribution check — verifies whether genres are balanced across the dataset.
# Imbalance can affect model performance and should be acknowledged.

genre_counts = df["genre"].value_counts()

colors = ["#2C3E50", "#2980B9", "#1ABC9C", "#8E44AD"]

plt.figure(figsize=(7, 4))
bars = plt.bar(genre_counts.index, genre_counts.values,
               color=colors, edgecolor="white", linewidth=1.2)

plt.title("Class Distribution Across Genres", fontsize=13, fontweight="bold", pad=12)
plt.xlabel("Genre", fontsize=11)
plt.ylabel("Number of Movies", fontsize=11)
plt.grid(axis="y", alpha=0.3, linestyle="--")
plt.gca().set_facecolor("#F8F9FA")

# Label on top of each bar
for bar, count in zip(bars, genre_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
             str(count), ha="center", va="bottom", fontweight="bold", fontsize=11)

plt.tight_layout()
plt.show()

# Imbalance ratio
print(f"Max/Min ratio: {genre_counts.max() / genre_counts.min():.2f}x")
print(genre_counts)

In [ ]:
# Sample movie posters from each genre — gives a visual understanding
# of the image data used in CNN-based models.

import os
from PIL import Image

poster_dir = "posters"

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle("Sample Movie Posters by Genre", fontsize=15, fontweight="bold", y=1.02)

genres = ["action", "drama", "comedy", "thriller"]

for col, genre in enumerate(genres):
    genre_movies = df[df["genre"] == genre]["tt_id"].values
    shown = 0
    for tt_id in genre_movies:
        path = os.path.join(poster_dir, f"{tt_id}.jpg")
        if os.path.exists(path):
            img = Image.open(path).convert("RGB")
            axes[shown][col].imshow(img)
            axes[shown][col].axis("off")
            if shown == 0:
                axes[shown][col].set_title(genre.capitalize(),
                                           fontsize=13,
                                           fontweight="bold",
                                           pad=8)
            shown += 1
        if shown == 2:
            break

plt.tight_layout()
plt.show()

## Preprocessing & Feature Engineering

This section prepares the scraped IMDb dataset for modeling across three data modalities: text, numerical, and image.

1. **Basic Cleaning** — rows with missing values are dropped, year is cast to integer, rating to float, and votes are converted from shorthand strings (e.g. "3.2M", "900K") to full numeric values.

2. **Label Encoding** — movie genres (action, comedy, drama, thriller) are encoded to integers using LabelEncoder for use as classification targets.

3. **Text Features** — movie titles are vectorized using TF-IDF (top 1000 features) to produce a sparse numerical representation of title vocabulary.

4. **Numerical Feature Scaling** — year, rating, and votes are normalized to [0, 1] using MinMaxScaler to ensure equal contribution across features with different ranges.

5. **Image Paths** — poster image paths are referenced by tt_id for use in CNN-based models (loaded directly during training via PyTorch Dataset).

6. **Final Feature Matrix** — TF-IDF text features and scaled numerical features are concatenated into a single matrix used as input for non-image models.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer

# Load dataset
df = pd.read_csv("imdb_clean.csv")

# -------------------------
# 1) Basic Cleaning (Safety)
# -------------------------
df = df.dropna()

# convert types
df["year"] = df["year"].astype(int)
df["rating"] = df["rating"].astype(float)

# clean votes (if still string like "3.2M")
def clean_votes(v):
    if isinstance(v, str):
        v = v.replace("(", "").replace(")", "").replace("M", "")
        try:
            return float(v) * 1_000_000
        except:
            return np.nan
    return v

df["votes"] = df["votes"].apply(clean_votes)
df = df.dropna()

# -------------------------
# 2) Encoding Genre
# -------------------------
le = LabelEncoder()
df["genre_encoded"] = le.fit_transform(df["genre"])

# -------------------------
# 3) Text Features (Title)
# -------------------------
tfidf = TfidfVectorizer(max_features=1000)
X_text = tfidf.fit_transform(df["title"]).toarray()

# -------------------------
# 4) Numerical Features Scaling
# -------------------------
scaler = MinMaxScaler()

num_features = df[["year", "rating", "votes"]]
X_num = scaler.fit_transform(num_features)

# -------------------------
# 5) Image Paths (just reference)
# -------------------------
X_img_paths = df["image"].values

# -------------------------
# 6) Final Feature Set (Non-image model example)
# -------------------------
X = np.hstack((X_text, X_num))
y = df["genre_encoded"]

print("Final X shape:", X.shape)
print("Labels shape:", y.shape)

## Models

This section trains 6 deep learning models on the scraped IMDb dataset, covering RNNs, CNNs, Transformers, and Vision Transformers across text, image, and multi-modal data.

- Model 1: Genre Classification from movie titles using a Bidirectional LSTM (RNN), with a second variant (1B) adding numerical metadata (year, rating, votes) to measure the impact of contextual features.
- Model 2: Genre Classification from movie poster images using a fine-tuned ResNet18 pretrained on ImageNet.
- Model 3: IMDb Rating Prediction using DistilBERT title embeddings (Transformer), evaluated with Linear Regression and Random Forest Regressor.
- Model 4: Era Detection — predicting the release decade from poster images using a fine-tuned ResNet18 with data augmentation.
- Model 5: Multi-Modal Genre Classification combining ResNet18 image features with scaled numerical metadata through a late fusion architecture.
- Model 6: Genre Classification from poster images using ViT-B/16 (Vision Transformer), compared directly against ResNet18 to evaluate CNN vs attention-based architectures on the same task and data.

In [ ]:
## Model 1: Genre Classification using LSTM on Movie Titles (RNN)
# We use an LSTM-based RNN to classify movie genres from their titles.
# LSTM is chosen over simple RNN because it handles vanishing gradients better
# and captures long-term dependencies in text sequences.
# This approach treats each title as a sequence of word embeddings,
# allowing the model to learn sequential patterns in movie naming conventions.

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from collections import Counter
import numpy as np

# ----------------------------
# Tokenizer & Vocabulary
# ----------------------------
def tokenize(text):
    return str(text).lower().split()

all_tokens = [token for title in df["title"] for token in tokenize(title)]
vocab = ["<PAD>", "<UNK>"] + [w for w, _ in Counter(all_tokens).most_common(5000)]
word2idx = {w: i for i, w in enumerate(vocab)}

MAX_LEN = 10

def encode(title):
    tokens = tokenize(title)[:MAX_LEN]
    ids = [word2idx.get(t, 1) for t in tokens]
    ids += [0] * (MAX_LEN - len(ids))
    return ids

# ----------------------------
# Dataset
# ----------------------------
le = LabelEncoder()
df["label"] = le.fit_transform(df["genre"])

class TitleDataset(Dataset):
    def __init__(self, dataframe):
        self.data = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        x = torch.tensor(encode(row["title"]), dtype=torch.long)
        y = torch.tensor(row["label"], dtype=torch.long)
        return x, y

train_df, test_df = train_test_split(df, test_size=0.2,
                                      stratify=df["label"],
                                      random_state=42)

train_dataset = TitleDataset(train_df)
test_dataset  = TitleDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

# ----------------------------
# LSTM Model
# ----------------------------
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim,
                            num_layers=2,
                            batch_first=True,
                            dropout=0.3,
                            bidirectional=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)
        # Concatenate last hidden state from both directions
        out = torch.cat([hidden[-2], hidden[-1]], dim=1)
        return self.classifier(out)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model_lstm = LSTMClassifier(
    vocab_size=len(vocab),
    embed_dim=64,
    hidden_dim=128,
    num_classes=len(le.classes_)
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_lstm.parameters(), lr=0.001)

# ----------------------------
# Training
# ----------------------------
EPOCHS = 15

for epoch in range(EPOCHS):
    model_lstm.train()
    total_loss = 0
    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model_lstm(x_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f}")

# ----------------------------
# Evaluation
# ----------------------------
model_lstm.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for x_batch, y_batch in test_loader:
        x_batch = x_batch.to(device)
        outputs = model_lstm(x_batch)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y_batch.numpy())

print(f"\nLSTM Accuracy: {accuracy_score(all_labels, all_preds):.2f}")
print(classification_report(all_labels, all_preds, target_names=le.classes_))

In [ ]:
## Model 1B: LSTM + Metadata — adding year, rating, and votes alongside the title
## to test whether metadata improves genre classification over text alone.

# ----------------------------
# Enhanced LSTM with Metadata
# ----------------------------
class LSTMWithMeta(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, meta_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim,
                            num_layers=2,
                            batch_first=True,
                            dropout=0.3,
                            bidirectional=True)
        self.meta_fc = nn.Sequential(
            nn.Linear(meta_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2 + 32, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x, meta):
        embedded = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)
        lstm_out = torch.cat([hidden[-2], hidden[-1]], dim=1)
        meta_out = self.meta_fc(meta)
        combined = torch.cat([lstm_out, meta_out], dim=1)
        return self.classifier(combined)

# ----------------------------
# Updated Dataset with Metadata
# ----------------------------
scaler = MinMaxScaler()
df[["year_s", "rating_s", "votes_s"]] = scaler.fit_transform(
    df[["year", "rating", "votes"]]
)

class TitleMetaDataset(Dataset):
    def __init__(self, dataframe):
        self.data = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        x = torch.tensor(encode(row["title"]), dtype=torch.long)
        meta = torch.tensor(
            [row["year_s"], row["rating_s"], row["votes_s"]],
            dtype=torch.float32
        )
        y = torch.tensor(row["label"], dtype=torch.long)
        return x, meta, y

train_df2, test_df2 = train_test_split(df, test_size=0.2,
                                        stratify=df["label"],
                                        random_state=42)

train_loader2 = DataLoader(TitleMetaDataset(train_df2), batch_size=32, shuffle=True)
test_loader2  = DataLoader(TitleMetaDataset(test_df2),  batch_size=32, shuffle=False)

# ----------------------------
# Train Enhanced LSTM
# ----------------------------
model_lstm2 = LSTMWithMeta(
    vocab_size=len(vocab),
    embed_dim=64,
    hidden_dim=128,
    meta_dim=3,
    num_classes=len(le.classes_)
).to(device)

optimizer2 = torch.optim.Adam(model_lstm2.parameters(), lr=0.001)

for epoch in range(15):
    model_lstm2.train()
    total_loss = 0
    for x_batch, meta_batch, y_batch in train_loader2:
        x_batch = x_batch.to(device)
        meta_batch = meta_batch.to(device)
        y_batch = y_batch.to(device)
        optimizer2.zero_grad()
        outputs = model_lstm2(x_batch, meta_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer2.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/15 | Loss: {total_loss/len(train_loader2):.4f}")

# ----------------------------
# Evaluation
# ----------------------------
model_lstm2.eval()
all_preds2, all_labels2 = [], []

with torch.no_grad():
    for x_batch, meta_batch, y_batch in test_loader2:
        x_batch = x_batch.to(device)
        meta_batch = meta_batch.to(device)
        outputs = model_lstm2(x_batch, meta_batch)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        all_preds2.extend(preds)
        all_labels2.extend(y_batch.numpy())

print(f"\nLSTM + Metadata Accuracy: {accuracy_score(all_labels2, all_preds2):.2f}")
print(classification_report(all_labels2, all_preds2, target_names=le.classes_))

In [ ]:
## Model 1 Summary — Comparison between LSTM variants

results_m1 = {
    "LSTM (Title Only)":        0.23,
    "LSTM + Metadata":          0.35,
}

plt.figure(figsize=(7, 4))
bars = plt.bar(results_m1.keys(), results_m1.values(),
               color=["#2980B9", "#1ABC9C"], edgecolor="white", linewidth=1.2)
plt.title("Model 1: LSTM Accuracy Comparison", fontsize=13, fontweight="bold")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.grid(axis="y", alpha=0.3, linestyle="--")
plt.gca().set_facecolor("#F8F9FA")

for bar, val in zip(bars, results_m1.values()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f"{val:.0%}", ha="center", fontweight="bold", fontsize=12)

plt.tight_layout()
plt.show()

print("Key Insight: Adding metadata (year, rating, votes) improved LSTM accuracy")
print("from 23% to 35%, confirming that movie titles alone carry limited genre signal.")

In [ ]:
## Model 2: Genre Classification (Image)
# Predict movie genre from poster images using a pre-trained ResNet18 with fine-tuning.

import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split
import numpy as np

poster_dir = "posters"
poster_files = {f.split(".")[0]: f for f in os.listdir(poster_dir)}

df_img = df[df["tt_id"].isin(poster_files.keys())].copy()
print(f"Movies with posters: {len(df_img)}")

le = LabelEncoder()
df_img["label"] = le.fit_transform(df_img["genre"])

class PosterDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.data = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_path = os.path.join(poster_dir, poster_files[row["tt_id"]])
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, row["label"]

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_df, test_df = train_test_split(df_img, test_size=0.2, random_state=42)

train_dataset = PosterDataset(train_df, transform=transform)
test_dataset = PosterDataset(test_df, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = models.resnet18(weights="IMAGENET1K_V1")
model.fc = nn.Linear(model.fc.in_features, len(le.classes_))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

for epoch in range(10):
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/10 | Loss: {total_loss/len(train_loader):.4f}")

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print(f"\nAccuracy: {accuracy_score(all_labels, all_preds):.2f}")
print(classification_report(all_labels, all_preds, target_names=le.classes_))

In [ ]:
## Model 3: Rating Prediction using BERT Embeddings (Transformer)
## We use DistilBERT as a feature extractor to generate title embeddings,
## then feed them into a regression model to predict IMDb ratings.
## BERT is chosen over TF-IDF because it captures semantic meaning of words,
## not just word frequency — making it a stronger text representation.

from transformers import DistilBertTokenizer, DistilBertModel
import torch
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

# ----------------------------
# Load DistilBERT
# ----------------------------
print("Loading DistilBERT...")
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
bert_model = DistilBertModel.from_pretrained("distilbert-base-uncased")
bert_model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bert_model = bert_model.to(device)
print(f"Device: {device}")

# ----------------------------
# Extract BERT Embeddings
# ----------------------------
def get_bert_embedding(titles, batch_size=32):
    all_embeddings = []
    for i in range(0, len(titles), batch_size):
        batch = list(titles[i:i+batch_size])
        encoded = tokenizer(batch, padding=True, truncation=True,
                           max_length=16, return_tensors="pt")
        encoded = {k: v.to(device) for k, v in encoded.items()}
        with torch.no_grad():
            output = bert_model(**encoded)
        # Use [CLS] token embedding as sentence representation
        embeddings = output.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(embeddings)
        if i % 100 == 0:
            print(f"  Processed {i}/{len(titles)} titles...")
    return np.vstack(all_embeddings)

print("\nExtracting BERT embeddings...")
X_bert = get_bert_embedding(df["title"].values)
print(f"Embeddings shape: {X_bert.shape}")

# ----------------------------
# Add Metadata
# ----------------------------
scaler = MinMaxScaler()
X_num = scaler.fit_transform(df[["year", "votes"]])
X = np.hstack((X_bert, X_num))
y = df["rating"]

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                      test_size=0.2,
                                                      random_state=42)

# ----------------------------
# Linear Regression
# ----------------------------
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
print("\nLinear Regression (BERT features):")
print(f"  MAE: {mean_absolute_error(y_test, y_pred_lr):.3f}")
print(f"  R2:  {r2_score(y_test, y_pred_lr):.3f}")

# ----------------------------
# Random Forest Regressor
# ----------------------------
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print("\nRandom Forest (BERT features):")
print(f"  MAE: {mean_absolute_error(y_test, y_pred_rf):.3f}")
print(f"  R2:  {r2_score(y_test, y_pred_rf):.3f}")

# ----------------------------
# Comparison Summary
# ----------------------------
print("\n--- Comparison: TF-IDF vs BERT embeddings ---")
print(f"{'Model':<30} {'MAE':>6} {'R2':>6}")
print("-" * 45)
print(f"{'Linear Regression (BERT)':<30} {mean_absolute_error(y_test, y_pred_lr):>6.3f} {r2_score(y_test, y_pred_lr):>6.3f}")
print(f"{'Random Forest (BERT)':<30} {mean_absolute_error(y_test, y_pred_rf):>6.3f} {r2_score(y_test, y_pred_rf):>6.3f}")

In [ ]:
## Model 4: Era Detection
# Predict the release decade of a movie from its poster using a fine-tuned ResNet18.

import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split
import numpy as np

poster_dir = "posters"
poster_files = {f.split(".")[0]: f for f in os.listdir(poster_dir)}

df_era = df[df["tt_id"].isin(poster_files.keys())].copy()

df_era["decade"] = (df_era["year"] // 10 * 10).astype(str) + "s"
print(df_era["decade"].value_counts())

le = LabelEncoder()
df_era["label"] = le.fit_transform(df_era["decade"])

class PosterDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.data = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_path = os.path.join(poster_dir, poster_files[row["tt_id"]])
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, row["label"]

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_df, test_df = train_test_split(df_era, test_size=0.2, random_state=42)

train_dataset = PosterDataset(train_df, transform=transform_train)
test_dataset = PosterDataset(test_df, transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = models.resnet18(weights="IMAGENET1K_V1")
model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model.fc.in_features, len(le.classes_))
)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.00005)

for epoch in range(5):
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/5 | Loss: {total_loss/len(train_loader):.4f}")

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

unique_labels = sorted(set(all_labels))
print(f"\nAccuracy: {accuracy_score(all_labels, all_preds):.2f}")
print(classification_report(all_labels, all_preds, labels=unique_labels, target_names=le.classes_[unique_labels]))

In [ ]:
## Model 5: Multi-Modal Genre Classification
# Improved Final Version

import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split
import numpy as np

poster_dir = "posters"
poster_files = {f.split(".")[0]: f for f in os.listdir(poster_dir)}

# keep only rows with downloaded posters
df_mm = df[df["tt_id"].isin(poster_files.keys())].copy().reset_index(drop=True)

# encode labels
le = LabelEncoder()
df_mm["label"] = le.fit_transform(df_mm["genre"])

# scale metadata
scaler = MinMaxScaler()
df_mm[["year_s", "rating_s", "votes_s"]] = scaler.fit_transform(
    df_mm[["year", "rating", "votes"]]
)

# -----------------------------
# Dataset Class
# -----------------------------
class MultiModalDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.data = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        img_path = os.path.join(
            poster_dir,
            poster_files[row["tt_id"]]
        )

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        meta = torch.tensor(
            [
                row["year_s"],
                row["rating_s"],
                row["votes_s"]
            ],
            dtype=torch.float32
        )

        return image, meta, row["label"]

# -----------------------------
# Better Transforms
# -----------------------------
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

# -----------------------------
# Better Split (important)
# -----------------------------
train_df, test_df = train_test_split(
    df_mm,
    test_size=0.2,
    stratify=df_mm["label"],
    random_state=42
)

train_dataset = MultiModalDataset(
    train_df,
    transform=transform_train
)

test_dataset = MultiModalDataset(
    test_df,
    transform=transform_test
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

# -----------------------------
# Multi-Modal Model
# -----------------------------
class MultiModalModel(nn.Module):
    def __init__(self, num_classes, meta_dim=3):
        super().__init__()

        self.resnet = models.resnet18(
            weights="IMAGENET1K_V1"
        )

        img_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Identity()

        self.meta_fc = nn.Sequential(
            nn.Linear(meta_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        self.classifier = nn.Sequential(
            nn.Linear(img_features + 64, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, image, meta):
        img_out = self.resnet(image)
        meta_out = self.meta_fc(meta)

        combined = torch.cat(
            [img_out, meta_out],
            dim=1
        )

        return self.classifier(combined)

# -----------------------------
# Device
# -----------------------------
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Device: {device}")

model = MultiModalModel(
    num_classes=len(le.classes_)
).to(device)

criterion = nn.CrossEntropyLoss()

# lower learning rate = better stability
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.00001
)

# -----------------------------
# Training
# -----------------------------
EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for images, meta, labels in train_loader:
        images = images.to(device)
        meta = meta.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images, meta)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Loss: {total_loss / len(train_loader):.4f}"
    )

# -----------------------------
# Evaluation
# -----------------------------
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, meta, labels in test_loader:
        images = images.to(device)
        meta = meta.to(device)

        outputs = model(images, meta)

        preds = torch.argmax(
            outputs,
            dim=1
        ).cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print("\nFinal Results")
print(
    f"Accuracy: "
    f"{accuracy_score(all_labels, all_preds):.2f}"
)

print(
    classification_report(
        all_labels,
        all_preds,
        target_names=le.classes_
    )
)

In [ ]:
print(df_img.shape)
print(le.classes_)
print(poster_dir)

In [ ]:
# ----------------------------------------
# 2) Failure Examples — using all_labels / all_preds directly
# ----------------------------------------
wrong_examples = []

for i in range(len(all_labels)):
    if all_labels[i] != all_preds[i]:
        wrong_examples.append({
            "true": le.classes_[all_labels[i]],
            "pred": le.classes_[all_preds[i]],
        })
    if len(wrong_examples) >= 8:
        break

print(f"\nWrong predictions sample ({len(wrong_examples)} shown):")
html = "<div style='display:flex; flex-wrap:wrap; gap:10px; margin-top:10px;'>"
for ex in wrong_examples[:8]:
    html += f"""
    <div style='border:2px solid #e74c3c; border-radius:8px; padding:10px;
                background:#ffcccc; width:160px; font-family:Arial; font-size:12px;'>
        <b>Actual:</b> {ex['true']}<br>
        <b>Predicted:</b> {ex['pred']}
    </div>"""
html += "</div>"
display(HTML(html))

print("""
Reflection:
- Action and Thriller share dark visual tones → model confuses them most.
- Comedy lacks strong visual patterns → often misclassified as Drama.
- Dataset size (822 movies) limits CNN generalization.
- Single-genre labels don't reflect multi-genre reality.
- Future: use CLIP or expand dataset for better results.
""")

In [ ]:
## Model 6: Vision Transformer (ViT-B/16) — Genre Classification from Poster Images
## ViT splits each poster into fixed patches and applies self-attention across all patches,
## allowing the model to capture global visual relationships unlike CNN's local filters.
## We compare it directly against ResNet18 (Model 2) on the same data and split.

from torchvision.models import vit_b_16, ViT_B_16_Weights
from sklearn.metrics import accuracy_score, classification_report

# Load Pretrained ViT
vit_model = vit_b_16(weights=ViT_B_16_Weights.DEFAULT)
vit_model.heads.head = nn.Linear(
    vit_model.heads.head.in_features,
    len(le.classes_)
)
vit_model = vit_model.to(device)

# Freeze all layers except the head
for param in vit_model.parameters():
    param.requires_grad = False
vit_model.heads.head.weight.requires_grad = True
vit_model.heads.head.bias.requires_grad = True

# DataLoaders
vit_train_loader = DataLoader(
    PosterDataset(train_df, transform=transform),
    batch_size=8, shuffle=True
)
vit_test_loader = DataLoader(
    PosterDataset(test_df, transform=transform),
    batch_size=8, shuffle=False
)

# Training
vit_optimizer = torch.optim.Adam(vit_model.heads.head.parameters(), lr=1e-3)
vit_criterion = nn.CrossEntropyLoss()

print("Training ViT-B/16 (head only)...\n")
for epoch in range(2):
    vit_model.train()
    total_loss = 0
    for images, labels in vit_train_loader:
        images, labels = images.to(device), labels.to(device)
        vit_optimizer.zero_grad()
        outputs = vit_model(images)
        loss = vit_criterion(outputs, labels)
        loss.backward()
        vit_optimizer.step()
        total_loss += loss.item()
    print(f"  Epoch {epoch+1}/2 | Loss: {total_loss/len(vit_train_loader):.4f}")

# Evaluation
vit_model.eval()
vit_preds, vit_labels_list = [], []

with torch.no_grad():
    for images, labels in vit_test_loader:
        images = images.to(device)
        preds = torch.argmax(vit_model(images), dim=1).cpu().numpy()
        vit_preds.extend(preds)
        vit_labels_list.extend(labels.numpy())

vit_acc    = accuracy_score(vit_labels_list, vit_preds)
resnet_acc = accuracy_score(all_labels, all_preds)

print("\n" + "="*45)
print("       ResNet18 vs ViT-B/16 — Accuracy")
print("="*45)
print(f"  ResNet18  →  {resnet_acc*100:.2f}%")
print(f"  ViT-B/16  →  {vit_acc*100:.2f}%")
print("="*45)

print("\nViT-B/16 Classification Report:")
print(classification_report(vit_labels_list, vit_preds, target_names=le.classes_))

## Model Selection & Architectural Justification

Each model in this project was selected to match a specific data modality and task type, covering the five core deep learning paradigms: RNNs, CNNs, Transformers, ViTs, and multi-modal fusion.

---

**Model 1 — Bidirectional LSTM (RNN)**
Task: Genre classification from movie titles.
Movie titles are short text sequences. LSTM is chosen over simple RNN because it solves the vanishing gradient problem and captures dependencies across words. The bidirectional variant reads each title forward and backward, giving richer context. A second variant (1B) adds year, rating, and votes to test whether metadata improves text-based classification — accuracy improved from 23% to 35%, confirming metadata carries genre signal.

---

**Model 2 — ResNet18 (Pretrained CNN)**
Task: Genre classification from poster images.
ResNet18 is chosen for its residual connections that enable deep feature extraction without degradation. Pretrained on ImageNet, it brings strong visual priors to a small dataset. Fine-tuning only the final layer keeps training fast while leveraging learned low-level features like edges, textures, and color patterns.

---

**Model 3 — DistilBERT (Transformer)**
Task: IMDb rating prediction from movie titles.
TF-IDF treats words independently and ignores context. DistilBERT uses attention mechanisms to understand contextual meaning — the word "dark" signals different things in a thriller versus a comedy. The [CLS] token embedding is used as a sentence-level representation, fed into Linear Regression and Random Forest for rating prediction.

---

**Model 4 — ResNet18 with Data Augmentation (Pretrained CNN)**
Task: Era detection — predicting release decade from poster.
Same architecture as Model 2, but with stronger augmentation (random flip, rotation, color jitter) to help the model generalize across decade-specific visual styles. Era detection achieved 67% accuracy — the highest among image models — because decade-level visual differences in posters are more visually distinct than genre differences.

---

**Model 5 — Multi-Modal Fusion (CNN + Metadata)**
Task: Genre classification combining image and numerical features.
A late fusion architecture combining ResNet18 image features with scaled metadata (year, rating, votes) through a fully connected layer. The motivation is that neither image nor metadata alone is sufficient — combining them should reduce ambiguity. Results showed limited improvement over single-modality baselines, likely due to small dataset size.

---

**Model 6 — ViT-B/16 (Vision Transformer)**
Task: Genre classification from poster images — compared against ResNet18.
Unlike CNNs that apply local filters across the image, ViT divides each poster into 16×16 patches and applies self-attention across all patches simultaneously, capturing global spatial relationships. Both ViT-B/16 and ResNet18 achieved similar accuracy (~39%), showing that with limited data, the global attention of ViT and the local inductive bias of CNN converge to similar performance.

## Conclusion

Text-based models outperformed image-based models for genre classification due to limited training data. Era Detection achieved the highest accuracy (67%) as decade-specific visual styles in posters are more distinguishable. Multi-modal fusion did not improve results, suggesting that with limited data, image features add noise rather than value.

Future work could address these limitations by expanding the dataset beyond 822 movies, incorporating additional metadata such as director and cast, and leveraging larger pre-trained vision-language models such as CLIP for improved multi-modal performance.